<a href="https://colab.research.google.com/github/NEFAxRAJ/Machine-Learning-Practices/blob/develop/Spam_Detection_using_Logistic_Regression_mathematical_intuition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Write a python code to implement a Logistic Regression Model for Spam detection**

Logistic Regression is a supervised learning algorithm used for classification, mainly when the outcome is binary (e.g., yes/no, 0/1).

It predicts the probability of a class using the sigmoid (logistic) function

Output lies between 0 and 1

A threshold (usually 0.5) converts probability into a class label. That is >= 0.5 rounds off to 1, otherwise it rounds off to 0.


$$z= w_1 x_1 + w_2 x_2 + \cdots +w_n x_n +c $$

$$ w_1...w_n → \text {Depends on number of features } $$


$$ \text{hypothesis } h_i \text{ = } \sigma(z)= \frac{1}{1+e^{-z}} $$

  Loss Function $$ L(y, ̂\hat{y}) = -\frac{1}{m} \sum_{i=1}^{m} [y \log(h_i) + (1-y)\log(1-h_i)]   $$

Gradient Descent for minimizing the loss
$$ w= w - \alpha \nabla_w J $$

$$ b= b - \alpha \nabla_b J $$

Gradient Function

$$ \nabla_w J = \frac{1}{m} \sum_{i=1}^{m} (h_i -y_i)(x_i) $$

$$ \nabla_b J = \frac{1}{m} \sum_{i=1}^{m} (h_i -y_i) $$










In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

Creating the Logistic Regression Model

In [2]:
#Cost or Log Loss function
def costfunc(w,x,y,c):
  h=hypothesis(w,x,c)
  cost= -y*np.log(h)-(1-y)*np.log(1-h)
  cost=np.mean(cost)
  return cost

In [3]:
def hypothesis(w,x,c):
  z=np.dot(x,w)+c
  h=sigmoid(z)
  return h

def sigmoid(z):
  z=1/(1+np.exp(-z))
  return z

In [4]:

def gradient_descent(x,y,learning_rate):
  m,n=x.shape
  w=np.zeros(n)
  c=0
  tcost=0
  for i in range(5000):
    new_w,new_c=gradientfunction(w,c,x,y,m)
    w=w-learning_rate*(new_w)
    c=c-learning_rate*(new_c)
    if(i%500==0):
      tcost=costfunc(w,x,y,c)
      print('Cost at ',i,' th interation is: ', tcost)
  return w,c;

def gradientfunction(w,c,x,y,m):
  err=hypothesis(w,x,c)-y
  new_w=np.sum(np.dot(x.T,err))
  new_c=np.sum(err)
  return new_w/m,new_c/m

*Fetching the Dataset*

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
df=pd.read_csv('/content/drive/MyDrive/sms_spam.csv')
dataset= pd.DataFrame(df)
data= pd.DataFrame(df)
data.head(5)

,type,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


Pre-processing the Dataset

In [7]:
data.isnull().sum()

,0
type,0
text,0


In [8]:
# Label ham= 0 and spam= 1 using labelencoder
from sklearn.preprocessing import LabelEncoder

data['type']= LabelEncoder().fit_transform(data['type'])
data.head(2)

,type,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...


In [9]:
# # Label ham= 10 and spam= 11 manually

# type_label={
#     'ham': 10,
#     'spam' : 11
# }
# dataset['type']= dataset['type'].map(type_label)
# dataset.head(2)

In [10]:
# Checking and removing duplicate values
data.duplicated().sum()

np.int64(414)

In [11]:
data=data.drop_duplicates(keep='first')
data.duplicated().sum()

np.int64(0)

In [12]:
## Data Pre-processing
    #1. Lower Case
    #2. Tokenization
    #3. Removing special characters
    #4. Removing stop words and punctuation
    #5. Stemming

In [13]:
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
from nltk.corpus import stopwords
import string
from nltk.stem.porter import PorterStemmer as ps

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [14]:
def text_preprocess(text):
  text=text.lower()
  text= nltk.word_tokenize(text)
  y=[]
  for i in text:
    if(i not in stopwords.words('english') and i not in string.punctuation and i.isalnum()):
      y.append(ps().stem(i))
  return " ".join(y)

In [15]:
text="Hi how are you not dancing today?"
txt=text_preprocess(text)
print(txt)

hi danc today


In [16]:
#Creating a new column of transformed text
data['transformed_text']=data['text'].apply(text_preprocess)
data.head(3)

,type,text,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...


In [17]:
#Vectorizing the data
from sklearn.feature_extraction.text import TfidfVectorizer
x=TfidfVectorizer().fit_transform(data['transformed_text']).toarray()
y=data['type'].values

In [18]:
#Splitting the dataset into Train And Test Dataset
x_train,x_test,y_train,y_test=train_test_split(x,y, test_size=0.3,random_state=2)

**Training the model**

In [19]:
learning_rate=0.1
w,c=gradient_descent(x_train,y_train,learning_rate)
print(w,c)

Cost at  0  th interation is:  0.6167818965541222
Cost at  500  th interation is:  0.35814030054750823
Cost at  1000  th interation is:  0.32421680077870296
Cost at  1500  th interation is:  0.31065120823639747
Cost at  2000  th interation is:  0.30423420418154906
Cost at  2500  th interation is:  0.3008623056319587
Cost at  3000  th interation is:  0.29896494729856093
Cost at  3500  th interation is:  0.29784602837523794
Cost at  4000  th interation is:  0.2971636965753216
Cost at  4500  th interation is:  0.2967372039873773
[1.46825708 1.46825708 1.46825708 ... 1.46825708 1.46825708 1.46825708] -6.234536740952647


Predict function

In [20]:
def predict(x, w, b):
    z = np.dot(x, w) + b
    h = sigmoid(z)
    return (h >= 0.5).astype(int)

Training and Testing Accuracy

In [21]:
y_train_pred = predict(x_train, w, c)
train_accuracy = np.mean(y_train_pred == y_train)
print("Training Accuracy:", train_accuracy * 100, "%")

y_test_pred = predict(x_test, w, c)
test_accuracy = np.mean(y_test_pred == y_test)
print("Testing Accuracy:", test_accuracy * 100, "%")

Training Accuracy: 86.40642303433002 %
Testing Accuracy: 87.79069767441861 %


Evaluating the Model

In [22]:
TP = FP = TN = FN = 0
m,n=x_test.shape

for i in range(m):
    if y_test[i] == 1 and y_test_pred[i] == 1:
        TP += 1
    elif y_test[i] == 1 and y_test_pred[i] == 0:
        FN += 1
    elif y_test[i] == 0 and y_test_pred[i] == 0:
        TN += 1
    elif y_test[i] == 0 and y_test_pred[i] == 1:
        FP += 1

print("TP:", TP, "FP:", FP, "TN:", TN, "FN:", FN)


TP: 18 FP: 26 TN: 1341 FN: 163


Accuracy

In [23]:
accuracy = (TP + TN) / m
print("Accuracy:", accuracy)

Accuracy: 0.877906976744186


Precision for 1


In [24]:
if (TP + FP) == 0:
    precision = 0
else:
    precision = TP / (TP + FP)
print("Precision:", precision)

Precision: 0.4090909090909091


Recall for 1

In [25]:
if (TP + FN) == 0:
    recall = 0
else:
    recall = TP / (TP + FN)

print("Recall:", recall)

Recall: 0.09944751381215469


F1-score for 1


In [26]:
if (precision + recall) == 0:
    f1 = 0
else:
    f1 = 2 * (precision * recall) / (precision + recall)

print("F1 Score:", f1)

F1 Score: 0.16


In [27]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, classification_report

In [28]:
print(accuracy_score(y_test,y_test_pred))
print(confusion_matrix(y_test,y_test_pred))
print(classification_report(y_test,y_test_pred))

0.877906976744186
[[1341   26]
 [ 163   18]]
              precision    recall  f1-score   support

           0       0.89      0.98      0.93      1367
           1       0.41      0.10      0.16       181

    accuracy                           0.88      1548
   macro avg       0.65      0.54      0.55      1548
weighted avg       0.84      0.88      0.84      1548

